In [2]:
pip install ccxt ta

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.16.1 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.1 which is incompatible.
tensorflow-intel 2.16.1 requires tensorboard<2.17,>=2.16, but you have tensorboard 2.19.0 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip



  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/5.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.7 MB ? eta -:--:--
   --- ------------------------------------ 0.5/5.7 MB 558.9 kB/s eta 0:00:10
   --- ------------------------------------ 0.5/5.7 MB 558.9 kB/s eta 0:00:10
   ----- ---------------------------------- 0.8/5.7 MB 559.5 kB/s eta 0:00:09
   ----- ---------------------------------- 0.8/5.7 MB 559.5 kB/s eta 0:00:09
   ------- -------------------------------- 1.0/5.7 MB 585.1 kB/s eta 0:00:08
   ------- -------------------------------- 1.0/5.7 MB 585.1 kB/s eta 0:00:08
   --------- ------------------------------ 1.3/5.7 MB 573.6 kB/s eta 0:00:08
   ---

In [3]:
# 2️⃣ Imports et configuration
import ccxt
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import ta

# Paramètres globaux
PAIRS = ['BTC/USDT', 'ETH/USDT', 'SOL/USDT', 'BNB/USDT', 'DOGE/USDT']
TIMEFRAME = '1h'
LIMIT = 1000      # Nombre de bougies à récupérer
WINDOW_SIZE = 30  # Nombre de bougies pour chaque état
exchange = ccxt.binance()


In [4]:
# 3️⃣ Fonction de récupération des données OHLCV
def fetch_ohlcv(symbol, timeframe=TIMEFRAME, limit=LIMIT):
    """
    Récupère les données OHLCV pour `symbol` et retourne un DataFrame indexé par timestamp.
    """
    data = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)
    df = pd.DataFrame(data, columns=['timestamp','open','high','low','close','volume'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df.set_index('timestamp', inplace=True)
    return df


In [5]:
# 4️⃣ Fonction de calcul des indicateurs techniqu
# es
def compute_indicators(df):
    """
    Prend un DataFrame OHLCV et renvoie un DataFrame avec les indicateurs :
    close, SMA(10), EMA(10), RSI(14), MACD, Bollinger Bands (20), volume normalisé.
    """
    df = df.copy()
    # Moyennes mobiles
    df['sma10'] = ta.trend.SMAIndicator(df['close'], window=10).sma_indicator()
    df['ema10'] = ta.trend.EMAIndicator(df['close'], window=10).ema_indicator()
    # RSI
    df['rsi'] = ta.momentum.RSIIndicator(df['close'], window=14).rsi()
    # MACD
    macd = ta.trend.MACD(df['close'])
    df['macd'] = macd.macd()
    # Bollinger Bands
    bb = ta.volatility.BollingerBands(df['close'], window=20)
    df['bollinger_h'] = bb.bollinger_hband()
    df['bollinger_l'] = bb.bollinger_lband()
    # Volume normalisé
    df['vol_mean'] = df['volume'].rolling(window=20).mean()
    df['vol_std'] = df['volume'].rolling(window=20).std()
    df['volume_norm'] = (df['volume'] - df['vol_mean']) / df['vol_std']
    # Sélection des colonnes finales
    df = df[['close','sma10','ema10','rsi','macd','bollinger_h','bollinger_l','volume_norm']]
    df.dropna(inplace=True)
    return df


In [6]:
# 5️⃣ Génération des fenêtres d'états pour l'agent DQN
def get_state_windows(df, window_size=WINDOW_SIZE):
    """
    Découpe le DataFrame en séquences de taille `window_size` pour constituer les états.
    Retourne un tableau NumPy de forme (n_states, window_size, n_features).
    """
    states = []
    for start in range(len(df) - window_size + 1):
        window = df.iloc[start:start + window_size].values
        states.append(window)
    return np.array(states)


In [7]:
# 6️⃣ Exemple complet pour BTC/USDT
df_btc = fetch_ohlcv('BTC/USDT')
df_btc_ind = compute_indicators(df_btc)
state_windows = get_state_windows(df_btc_ind)

print(f"Shape des états (BTC/USDT) : {state_windows.shape}  ➡️  (n_states, {WINDOW_SIZE}, n_indicateurs)")


Shape des états (BTC/USDT) : (946, 30, 8)  ➡️  (n_states, 30, n_indicateurs)
